In [1]:
import torch
import torch.nn as nn

# Single neuron forward pass
x = torch.tensor([[90.0, 42.0, 43.0, 20.8, 82.0, 6.5, 202.0]])

model = nn.Sequential(
    nn.Linear(7, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 22)
)

out = model(x)
print('Output shape:', out.shape)          # [1, 22]
print('Predicted class:', out.argmax().item())

Output shape: torch.Size([1, 22])
Predicted class: 16


In [2]:
import torch

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print('Device:', device)
print('PyTorch version:', torch.__version__)

# Tensor basics
x = torch.zeros(32, 7)
print('Shape:', x.shape)
print('Dtype:', x.dtype)
print('Elements:', x.numel())

Device: cpu
PyTorch version: 2.11.0+cpu
Shape: torch.Size([32, 7])
Dtype: torch.float32
Elements: 224


In [3]:
import sys
sys.path.append('../utils')
from preprocess import get_dataloaders

train_loader, val_loader, classes = get_dataloaders()
print('Train batches:', len(train_loader))
print('Val batches  :', len(val_loader))
print('Classes      :', len(classes))

imgs, lbls = next(iter(train_loader))
print('Batch shape  :', imgs.shape)   # [32, 3, 128, 128]

ModuleNotFoundError: No module named 'config'

In [ ]:
import matplotlib.pyplot as plt
import torch

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t):
    return (t * STD + MEAN).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = denorm(imgs[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(classes[lbls[i]], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/batch_sample.png')
plt.show()

In [ ]:
from train import build_model, build_custom_cnn, device

num_classes = len(classes)

# Option A: ResNet18 with pretrained weights (recommended)
model = build_model(num_classes, use_pretrained=True)

# Option B: Custom CNN from scratch
# model = build_custom_cnn(num_classes)

print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params    : {total_params:,}')
print(f'Trainable params: {trainable:,}')

# Test forward pass
dummy = torch.randn(2, 3, 128, 128).to(device)
out   = model(dummy)
print(f'Output shape    : {out.shape}')  # [2, 38]

In [ ]:
from train import train, unfreeze_model

# Phase 1: train only the head (fast)
history = train(model, train_loader, val_loader, epochs=5, lr=0.001)

# Phase 2: fine-tune all layers (slower, higher accuracy)
model   = unfreeze_model(model)
history2 = train(model, train_loader, val_loader, epochs=15, lr=0.0001)

In [ ]:
from evaluate import plot_training_curves

# Combine both phases
combined_history = {
    'train_loss': history['train_loss'] + history2['train_loss'],
    'val_acc':    history['val_acc']    + history2['val_acc'],
}
plot_training_curves(combined_history)

In [ ]:
from evaluate import load_model, evaluate

model, classes = load_model('../models/disease_model.pth')
acc, top5      = evaluate(model, val_loader, classes)

In [ ]:
from predict import load_model, predict

model, classes = load_model('../models/disease_model.pth')
results        = predict('../data/test_leaf.jpg', model, classes, top_k=3)

for r in results:
    print(f"{r['class']:<35} {r['confidence']:>6.2f}%")